# Phase 2 — EEGNet + Attention using the same 5-fold pipeline

In [ ]:
"""
Only one thing changes in Phase 2:

old model: baseline EEGNet
new model: EEGNet + Attention

Everything else stays the same:

same subject
same filtered data
same 5 folds
same training function structure
same evaluation style
same final mean ± std

That is very important, because this makes the comparison fair.

Notebook 1 : 01_subjectwise_5fold_baseline_eegnet.ipynb -->  stays as clean baseline record
Notebook 2 : 02_subjectwise_5fold_attention_eegnet.ipynb --> becomes clean attention-model record

STEPS:
1. Define the attention model.
2. Define the one-fold training function for the attention model.
3. Test only Fold 1 first.
4. If Fold 1 works, run all 5 folds.
5. Compute mean ± std.
Then compare:
Baseline EEGNet mean ± std
Attention EEGNet mean ± std


From Phase 1:
Mean accuracy = 0.1282
Std = 0.0267
So this is reference.


"""

In [1]:
# ---------------------------------------
# step 1 : Mount Google Drive
# ---------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ---------------------------------------
# step 2: importing the required libraries
# ---------------------------------------


import os
import random
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, SeparableConv2D
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import Dropout, Flatten, Dense
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.utils import to_categorical

In [3]:
# ---------------------------------------
# step 3 : Basic experiment settings
# ---------------------------------------


random_seed = 42
subject_file_name = "S14_EEG.mat"

dataset_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14"
results_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/Results_April2026"

number_of_signal_columns = 24576
number_of_channels = 6
number_of_samples_per_trial = 4096
number_of_folds = 5

batch_size = 16
number_of_epochs = 50
validation_size_inside_training = 0.2

In [4]:
# ---------------------------------------
# step 4 : make results folder and fix randomness
# ---------------------------------------

# Creates the results folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

np.random.seed(random_seed)
random.seed(random_seed)
tf.random.set_seed(random_seed)
print("==============================================")
print("Results folder is ready.")
print("Random seed is set.")
print("==============================================")

Results folder is ready.
Random seed is set.


In [5]:
# ---------------------------------------
# step 5 : building the subject path and load the file
# ---------------------------------------

import scipy.io as sio

subject_file_name = "S14_EEG.mat"

subject_file_path = os.path.join(dataset_folder, subject_file_name)

print("==============================================")
print("Subject file path:", subject_file_path)
print("Does file exist?", os.path.exists(subject_file_path))
print("==============================================")
mat_data = sio.loadmat(subject_file_path)
print("==============================================")
print("\nThe .mat file loaded successfully.")
print("Available keys inside the file:", mat_data.keys())
print("==============================================")

Subject file path: /content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14/S14_EEG.mat
Does file exist? True

The .mat file loaded successfully.
Available keys inside the file: dict_keys(['__header__', '__version__', '__globals__', 'EEG'])


In [7]:
# ---------------------------------------
# step 6 : extract EEG matrix and printing shape
# ---------------------------------------

eeg_matrix = mat_data["EEG"]
print("==============================================")
print("EEG matrix extracted.")
print("number_of_trials, number_of_columns are : ")
print("==============================================")
print("EEG matrix shape:", eeg_matrix.shape)
print("==============================================")

EEG matrix extracted.
number_of_trials, number_of_columns are : 
EEG matrix shape: (639, 24579)


In [8]:
# ---------------------------------------
# step 7 : separating the signal data and metadata columns
# ---------------------------------------



number_of_signal_columns = 24576

signal_data = eeg_matrix[:, :number_of_signal_columns]
modality_column = eeg_matrix[:, number_of_signal_columns]
stimulus_column = eeg_matrix[:, number_of_signal_columns + 1]
artifact_column = eeg_matrix[:, number_of_signal_columns + 2]

print("==============================================")
print("Signal data shape:", signal_data.shape)
print("Modality column shape:", modality_column.shape)
print("Stimulus column shape:", stimulus_column.shape)
print("Artifact column shape:", artifact_column.shape)
print("==============================================")

print("Unique modality values:", np.unique(modality_column))
print("Unique stimulus values:", np.unique(stimulus_column))
print("Unique artifact values:", np.unique(artifact_column))
print("==============================================")


Signal data shape: (639, 24576)
Modality column shape: (639,)
Stimulus column shape: (639,)
Artifact column shape: (639,)
Unique modality values: [1. 2.]
Unique stimulus values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]
Unique artifact values: [1. 2.]


In [9]:
# ---------------------------------------
# step 8: filtering only imagined speech and valid trials
# ---------------------------------------

valid_trial_mask = (modality_column == 1) & (artifact_column == 1)

filtered_signal_data = signal_data[valid_trial_mask]
filtered_labels = stimulus_column[valid_trial_mask]

print("==============================================")
print("Number of valid filtered trials:", len(filtered_labels))
print("Filtered signal shape:", filtered_signal_data.shape)
print("Filtered labels shape:", filtered_labels.shape)
print("==============================================")

print("Unique filtered labels:", np.unique(filtered_labels))
print("==============================================")

Number of valid filtered trials: 351
Filtered signal shape: (351, 24576)
Filtered labels shape: (351,)
Unique filtered labels: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


In [10]:
# ---------------------------------------
# step 9: reshape filtered signal into trial format
# ---------------------------------------


number_of_channels = 6
number_of_samples_per_trial = 4096

X = filtered_signal_data.reshape(-1, number_of_channels, number_of_samples_per_trial)
y = filtered_labels.astype(int)

print("==============================================")
print("X shape after reshape:", X.shape)
print("y shape:", y.shape)
print("==============================================")

X shape after reshape: (351, 6, 4096)
y shape: (351,)


In [11]:
# ---------------------------------------
# step 10 : prepare labels for classification - shift labels to 0 ... 10
# ---------------------------------------

y = y - 1

number_of_classes = len(np.unique(y))
label_counts = pd.Series(y).value_counts().sort_index()

print("==============================================")
print("Unique labels after shifting to 0-based indexing:")
print(np.unique(y))
print("==============================================")

print("Number of classes:", number_of_classes)
print("==============================================")

print("Class counts:")
print(label_counts)
print("==============================================")

Unique labels after shifting to 0-based indexing:
[ 0  1  2  3  4  5  6  7  8  9 10]
Number of classes: 11
Class counts:
0     35
1     40
2     37
3     33
4     34
5     28
6     32
7     31
8     27
9     25
10    29
Name: count, dtype: int64


In [13]:
# ---------------------------------------
# step 11: creating 5-fold stratified cross-validation splits
# ---------------------------------------

from sklearn.model_selection import StratifiedKFold

number_of_folds = 5
random_seed = 42

stratified_kfold = StratifiedKFold(
    n_splits=number_of_folds,
    shuffle=True,
    random_state=random_seed
)

fold_splits = list(stratified_kfold.split(X, y))

print("==============================================")
print("Total number of folds created:", len(fold_splits))
print("==============================================")

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    print("Fold", fold_number)
    print("Train size:", len(train_indices))
    print("Test size:", len(test_indices))
    print("----------------------------------------------")


Total number of folds created: 5
Fold 1
Train size: 280
Test size: 71
----------------------------------------------
Fold 2
Train size: 281
Test size: 70
----------------------------------------------
Fold 3
Train size: 281
Test size: 70
----------------------------------------------
Fold 4
Train size: 281
Test size: 70
----------------------------------------------
Fold 5
Train size: 281
Test size: 70
----------------------------------------------


In [14]:
# Defining the attention model

# ==============================================
# Phase 2 - Step 12 : defining EEGNet + lightweight attention model
# ==============================================

from tensorflow.keras.layers import GlobalAveragePooling2D, Reshape, Multiply

def build_attention_eegnet(input_shape, number_of_classes):
    input_layer = Input(shape=input_shape)

    # ------------------------------------------
    # EEGNet backbone - Block 1
    # ------------------------------------------
    block_1 = Conv2D(
        filters=8,
        kernel_size=(1, 64),
        padding="same",
        use_bias=False
    )(input_layer)
    block_1 = BatchNormalization()(block_1)

    block_1 = DepthwiseConv2D(
        kernel_size=(input_shape[0], 1),
        depth_multiplier=2,
        use_bias=False,
        depthwise_constraint=max_norm(1.0)
    )(block_1)
    block_1 = BatchNormalization()(block_1)
    block_1 = Activation("elu")(block_1)
    block_1 = AveragePooling2D(pool_size=(1, 4))(block_1)
    block_1 = Dropout(0.5)(block_1)

    # ------------------------------------------
    # EEGNet backbone - Block 2
    # ------------------------------------------
    block_2 = SeparableConv2D(
        filters=16,
        kernel_size=(1, 16),
        padding="same",
        use_bias=False
    )(block_1)
    block_2 = BatchNormalization()(block_2)
    block_2 = Activation("elu")(block_2)
    block_2 = AveragePooling2D(pool_size=(1, 8))(block_2)
    block_2 = Dropout(0.5)(block_2)

    # ------------------------------------------
    # Lightweight channel attention
    # ------------------------------------------
    attention_weights = GlobalAveragePooling2D()(block_2)
    attention_weights = Dense(
        units=16,
        activation="sigmoid"
    )(attention_weights)
    attention_weights = Reshape((1, 1, 16))(attention_weights)

    attended_features = Multiply()([block_2, attention_weights])

    # ------------------------------------------
    # Classification head
    # ------------------------------------------
    flatten_layer = Flatten()(attended_features)

    output_layer = Dense(
        units=number_of_classes,
        activation="softmax",
        kernel_constraint=max_norm(0.25)
    )(flatten_layer)

    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [15]:
input_shape = (6, 4096, 1)

test_attention_model = build_attention_eegnet(
    input_shape=input_shape,
    number_of_classes=number_of_classes
)

test_attention_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 4096,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 6, 4096,   │        512 │ input_layer[0][0] │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 6, 4096,   │         32 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 1, 4096,   │         96 │ batch_normalizat… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 4096,   │         64 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 1, 4096,   │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling2d   │ (None, 1, 1024,   │          0 │ activation[0][0]  │
│ (AveragePooling2D)  │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1, 1024,   │          0 │ average_pooling2… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d    │ (None, 1, 1024,   │        512 │ dropout[0][0]     │
│ (SeparableConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1, 1024,   │         64 │ separable_conv2d… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 1, 1024,   │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling2d_1 │ (None, 1, 128,    │          0 │ activation_1[0][… │
│ (AveragePooling2D)  │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 1, 128,    │          0 │ average_pooling2… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ dropout_1[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        272 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 1, 16)  │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1, 128,    │          0 │ dropout_1[0][0],  │
│                     │ 16)               │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 24,091 (94.11 KB)

 Trainable params: 24,011 (93.79 KB)

 Non-trainable params: 80 (320.00 B)

In [ ]:
"""

This model keeps the same EEGNet backbone as before.
Then we add a small attention step.

attention block means:
I tells the model that, among the learned feature maps, some are more important than others.
Give more weight to useful ones and less weight to weak ones
So instead of treating every feature equally, the model tries to focus more on the useful learned EEG patterns.


this attention version is lightweight.

We are not adding a giant attention network.

We are adding:
-> a global summary of the feature maps
-> a small dense layer to generate attention weights
-> multiplication of those weights with the feature maps
This keeps the model simpler and safer for your small dataset.


It is only defining the model function.
This is just to check that the model builds without errors.



As per the output:

attention model was created successfully, and the total number of parameters is:
  -> 24,091 parameters
That is very important.

What the attention block is doing :
-> After EEGNet extracts features, the attention block does this:
    -> looks at the learned feature maps
    -> creates a small importance score for each feature channel
    -> then multiplies the feature maps by those scores
the model is trying to say,
“these learned patterns matter more, these matter less.”
This is why it is called attention.

NEXT:

Now we create the one-fold training function for the attention model.
This will be almost the same as in Phase 1.

The only major change is:
    -> before: build_baseline_eegnet(...)
    -> now: build_attention_eegnet(...)
Everything else stays the same.
That is how we make the comparison fair.


"""

In [18]:
# Create the one-fold training function

# ==============================================
# Phase 2 - Step 13 : function to train and evaluate one fold
# using the attention model
# ==============================================

def run_one_fold_attention(X, y, train_indices, test_indices, fold_number):
    # ------------------------------------------
    # Step 13.1: create training and test data
    # ------------------------------------------
    X_train_full = X[train_indices]
    y_train_full = y[train_indices]

    X_test = X[test_indices]
    y_test = y[test_indices]

    # ------------------------------------------
    # Step 13.2: create validation set from training only
    # ------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_size_inside_training,
        stratify=y_train_full,
        random_state=random_seed
    )

    # ------------------------------------------
    # Step 13.3: normalize using only training data
    # ------------------------------------------
    training_mean = np.mean(X_train)
    training_std = np.std(X_train)

    if training_std == 0:
        training_std = 1.0

    X_train = (X_train - training_mean) / training_std
    X_val = (X_val - training_mean) / training_std
    X_test = (X_test - training_mean) / training_std

    # ------------------------------------------
    # Step 13.4: add final dimension for CNN input
    # ------------------------------------------
    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # ------------------------------------------
    # Step 13.5: convert labels to one-hot format
    # ------------------------------------------
    y_train_one_hot = to_categorical(y_train, num_classes=number_of_classes)
    y_val_one_hot = to_categorical(y_val, num_classes=number_of_classes)
    y_test_one_hot = to_categorical(y_test, num_classes=number_of_classes)

    # ------------------------------------------
    # Step 13.6: build a fresh attention model
    # ------------------------------------------
    input_shape = (number_of_channels, number_of_samples_per_trial, 1)

    model = build_attention_eegnet(
        input_shape=input_shape,
        number_of_classes=number_of_classes
    )

    print("==============================================")
    print("Running attention model for fold number:", fold_number)
    print("Training shape:", X_train.shape)
    print("Validation shape:", X_val.shape)
    print("Test shape:", X_test.shape)
    print("==============================================")

    # ------------------------------------------
    # Step 13.7: train the model
    # ------------------------------------------
    history = model.fit(
        X_train,
        y_train_one_hot,
        epochs=number_of_epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val_one_hot),
        verbose=1
    )

    # ------------------------------------------
    # Step 13.8: evaluate on the test fold
    # ------------------------------------------
    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_one_hot,
        verbose=0
    )

    print("Attention model fold", fold_number, "test accuracy:", test_accuracy)
    print("==============================================")

    result_dictionary = {
        "fold_number": fold_number,
        "train_size": int(len(X_train)),
        "validation_size": int(len(X_val)),
        "test_size": int(len(X_test)),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy)
    }

    return result_dictionary, history

In [ ]:
"""

This function is almost the same as your baseline fold function.

It still:
-> splits train/test
-> creates validation from train only
-> normalizes using training data only
-> adds the CNN dimension
-> converts labels to one-hot
-> trains one model
-> tests one fold

The only real difference is:
It builds the attention model
using:
build_attention_eegnet(...)
instead of the baseline model.

"""

In [19]:
# run only Fold 1 first for the attention model


# ==============================================
# Phase 2 - Step 14 : test only Fold 1 first
# ==============================================

first_train_indices, first_test_indices = fold_splits[0]

attention_fold_1_result, attention_fold_1_history = run_one_fold_attention(
    X=X,
    y=y,
    train_indices=first_train_indices,
    test_indices=first_test_indices,
    fold_number=1
)

print("Attention Fold 1 finished.")
print(attention_fold_1_result)

Running attention model for fold number: 1
Training shape: (224, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 11s 400ms/step - accuracy: 0.0893 - loss: 2.4252 - val_accuracy: 0.0714 - val_loss: 2.3977
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 8s 548ms/step - accuracy: 0.5357 - loss: 2.2095 - val_accuracy: 0.0893 - val_loss: 2.3926
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 375ms/step - accuracy: 0.7411 - loss: 2.0496 - val_accuracy: 0.0893 - val_loss: 2.3827
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 12s 460ms/step - accuracy: 0.8393 - loss: 1.9198 - val_accuracy: 0.0714 - val_loss: 2.3716
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 11s 509ms/step - accuracy: 0.8393 - loss: 1.8297 - val_accuracy: 0.0893 - val_loss: 2.3596
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 9s 382ms/step - accuracy: 0.8795 - loss: 1.7395 - val_accuracy: 0.1071 - val_loss: 2.3482
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 10s 375ms/step - accuracy: 0.8661 - loss: 1.6925 -

In [ ]:
"""
First line : It takes the first fold from your already-created 5 folds.
So:
-> one training side
-> one testing side
-> Next part : run_one_fold_attention(...)

It runs the full attention-model pipeline for Fold 1:
-> split train and test
-> make validation split
-> normalize
-> add input dimension
-> one-hot encode
-> build fresh attention model
-> train it
-> evaluate on test fold
-> Final print

It prints the result dictionary.
So at the end, WE should see :
1. fold number
2. train size
3. validation size
4. test size
5. test loss
6. test accuracy


As per the output:


The full Phase 2 pipeline is functioning:

-> data split worked
-> validation split worked
-> normalization worked
-> attention model built correctly
-> training ran fully
-> test evaluation worked


Fold 1 attention result :

we got:
  -> Attention Fold 1 test accuracy = 0.1549
  -> that is about 15.49%
Compare with baseline Fold 1

Earlier, your baseline Fold 1 was about:
  -> 21.13% in the first isolated Fold 1 test run you showed earlier
But in the final 5-fold baseline table, Fold 1 became: 12.68%

So the cleaner comparison should be made only after we run all 5 folds for attention,
not from one single fold alone.
which is very important.

the training behavior shows : Just like baseline,
the attention model is also showing:
    -> very high training accuracy
    -> much lower validation accuracy
    -> modest test accuracy

So overfitting is still there.
  " attention alone may help a little, or may not help enough by itself "

And that is exactly why later:
  -> augmentation
  -> transfer learning
  -> better evaluation  will matter.

The real comparison must be:
    -->> mean accuracy across all 5 folds
    -->> standard deviation across all 5 folds
Only then we can say whether attention helped or not.

"""

In [20]:
# ==============================================
# Phase 2 - Step 15 : run all 5 folds for attention model
# ==============================================

all_attention_fold_results = []
all_attention_fold_histories = []

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    fold_result, fold_history = run_one_fold_attention(
        X=X,
        y=y,
        train_indices=train_indices,
        test_indices=test_indices,
        fold_number=fold_number
    )

    all_attention_fold_results.append(fold_result)
    all_attention_fold_histories.append(fold_history)

print("==============================================")
print("All 5 attention-model folds finished.")
print("==============================================")

Running attention model for fold number: 1
Training shape: (224, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 11s 555ms/step - accuracy: 0.1027 - loss: 2.4293 - val_accuracy: 0.1250 - val_loss: 2.3950
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 378ms/step - accuracy: 0.5804 - loss: 2.1796 - val_accuracy: 0.1607 - val_loss: 2.3894
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 6s 422ms/step - accuracy: 0.7902 - loss: 2.0105 - val_accuracy: 0.1964 - val_loss: 2.3793
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 10s 389ms/step - accuracy: 0.8304 - loss: 1.8812 - val_accuracy: 0.2143 - val_loss: 2.3692
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 536ms/step - accuracy: 0.8438 - loss: 1.7969 - val_accuracy: 0.1964 - val_loss: 2.3590
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 378ms/step - accuracy: 0.8929 - loss: 1.7129 - val_accuracy: 0.1964 - val_loss: 2.3504
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 12s 506ms/step - accuracy: 0.8616 - loss: 1.6488 - 

In [21]:
# Convert attention results into a table

# ==============================================
# Step 16: convert attention fold results into table
# ==============================================

attention_results_dataframe = pd.DataFrame(all_attention_fold_results)

print("==============================================")
print("Attention model fold-wise results:")
print("==============================================")
display(attention_results_dataframe)

Attention model fold-wise results:


,fold_number,train_size,validation_size,test_size,test_loss,test_accuracy
0,1,224,56,71,2.506601,0.140845
1,2,224,57,70,2.481681,0.157143
2,3,224,57,70,2.752830,0.085714
3,4,224,57,70,2.656942,0.157143
4,5,224,57,70,2.615015,0.085714


In [24]:
# Compute final mean and standard deviation

# ==============================================
# Step 17 : calculate final mean and standard deviation
# ==============================================

attention_mean_accuracy = attention_results_dataframe["test_accuracy"].mean()
attention_std_accuracy = attention_results_dataframe["test_accuracy"].std()

print("==============================================")
print("Final ATTENTION EEGNet result")
print("==============================================")
print("Fold accuracies:", attention_results_dataframe["test_accuracy"].tolist())
print("Mean accuracy:", attention_mean_accuracy)
print("Standard deviation:", attention_std_accuracy)
print("Mean ± Std:", str(round(attention_mean_accuracy, 4)) + " ± " + str(round(attention_std_accuracy, 4)))
print("==============================================")

Final ATTENTION EEGNet result
Fold accuracies: [0.14084507524967194, 0.15714286267757416, 0.08571428805589676, 0.15714286267757416, 0.08571428805589676]
Mean accuracy: 0.12531187534332275
Standard deviation: 0.03675473318719575
Mean ± Std: 0.1253 ± 0.0368


In [23]:
"""
FINAL COMPARISON :

Baseline EEGNet (Phase 1)
Mean: 0.1282
Std: 0.0267
Attention EEGNet (Phase 2)
Mean: 0.1253
Std: 0.0368


1. attention mechanism DIDN'T improve performance. Because, if we see
-> Baseline: 12.82%
-> Attention: 12.53%
So attention is slightly worse, not better.

2. Is this a failure?
NO. This is actually GOOD for research.
In research, proving something does NOT work is also valuable.
we have just shown: “Adding attention alone does not solve the problem.”
That is a valid scientific result.

3. stability?
Compare standard deviation:
-> Baseline std: 2.67%
-> Attention std: 3.68%
Attention model is more unstable. This means:
-> performance varies more across folds
-> model is less consistent

4. Overfitting still exists
From training logs (same pattern):
Training accuracy ->  very high
Validation ->  low
Test ->  low
So attention did NOT fix overfitting

*** Simply adding attention does not improve imagined speech EEG classification under low-data conditions. ***

*** “The attention-enhanced EEGNet model achieved a mean accuracy of 12.53% ± 3.68%,
which is slightly lower than the baseline EEGNet performance (12.82% ± 2.67%).
This indicates that the addition of attention alone does not improve generalization
under limited data conditions and may increase model instability.” ***

attention did NOT help. because of the below reasons:

Reason 1 — Not enough data
we only have: ~351 trials
Attention needs more data to learn meaningful “importance”.
With small data, attention can:
    -->> learn noise instead of signal
    -->> worsen generalization

Reason 2 — Model complexity increased
Even though we kept it lightweight:
  -->> baseline ~23K parameters
  -->> attention ~24K parameters
Still slightly more complex
More complexity + small data = worse generalization

Reason 3 — EEG is already noisy
Imagined speech EEG:
  -->> weak signal
  -->> high noise
  -->> high variability across trials
Attention cannot easily “focus” on stable patterns because patterns themselves are unstable

Data-to-parameter ratio is bad → severe overfitting -->> Proved experimentally.

Next, What is the real solution?
Attention is NOT the main solution
The real solution is:
DATA + TRAINING STRATEGY

Specifically:
1. Data Augmentation (NEXT STEP)
To increase effective training samples

2. Transfer Learning (VERY IMPORTANT)
To learn from all subjects first

project direction is now clear. now we have:
-> Phase 1 : Baseline -> weak performance
-> Phase 2 : Attention -> no improvement
-> Phase 3 (NEXT) : Data Augmentation



Problem:
  -->> Small EEG dataset
  -->> Severe overfitting
  -->> Poor generalization

Experiment 1: Baseline EEGNet -> weak performance
Experiment 2: Attention + EEGNet -> no improvement
Insight: Architectural changes alone are insufficient

Proposed direction:
Improve data and training strategy instead
What we do next
We now move to:

PHASE 3 — DATA AUGMENTATION
We will start very simple:
Step 1:
Add Gaussian noise augmentation

because, this is
-> simplest
-> afest
-> easy to debug


Important rule : We will NOT:
    -> change model
    -> change folds
    -> change preprocessing

We will ONLY: add augmentation


"""